# Del microdato al indicador: preparación de los microdatos de la GEIH

Este cuaderno se ejecuta una vez por cada mes nuevo, en Google Colab o en Visual Studio Code. Lee los archivos que se descargan del DANE y genera los tres archivos CSV que se cargan en la base de datos (persona, ocupado y desocupado).

**Estructura de carpetas**

```
Proyecto GEIH/
├── Microdatos GEIH/        archivos descargados del DANE
│   └── Año/
│       ├── Mes/
└── Datos para BD/          archivos listos para cargar en MySQL
    └── Año/
        ├── Mes/
```

Cada mes se descarga en su carpeta de `Microdatos GEIH` y el resultado queda en la carpeta del mismo año y mes de `Datos para BD`.

## 1. Librerías y acceso a las carpetas

In [1]:
import os
import pandas as pd

# En Colab se conecta Google Drive; en otro entorno esta celda no hace nada
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

## 2. Parámetros

Lo único que se cambia cada mes es el año y el mes (y la carpeta raíz si se mueve el proyecto).

In [47]:
ANIO = 2026
MES = 3
RAIZ = r'C:\Users\jessi\OneDrive\Documents\GitHub\Proyecto-ingeniera-de-datos-\Entregable 2'    # carpeta que contiene las dos carpetas del proyecto

NOMBRES_MES = ['Enero', 'Febrero', 'Marzo', 'Abril', 'Mayo', 'Junio',
               'Julio', 'Agosto', 'Septiembre', 'Octubre', 'Noviembre', 'Diciembre']
id_periodo = ANIO * 100 + MES

CARPETA_ENTRADA = os.path.join(RAIZ, 'Microdatos GEIH', str(ANIO), NOMBRES_MES[MES - 1])
CARPETA_SALIDA = os.path.join(RAIZ, 'Datos para BD', str(ANIO), NOMBRES_MES[MES - 1])
os.makedirs(CARPETA_SALIDA, exist_ok=True)

# Nombres de los archivos del mes (cada módulo viene en un solo archivo)
ARCHIVOS = {
    'personas':    'Características generales, seguridad social en salud y educación.CSV',
    'ocupados':    'Ocupados.CSV',
    'no_ocupados': 'No ocupados.CSV',
}

print('Entrada:', CARPETA_ENTRADA)
print('Archivos encontrados:', os.listdir(CARPETA_ENTRADA))

Entrada: C:\Users\jessi\OneDrive\Documents\GitHub\Proyecto-ingeniera-de-datos-\Entregable 2\Microdatos GEIH\2026\Marzo
Archivos encontrados: ['Características generales, seguridad social en salud y educación.CSV', 'No ocupados.CSV', 'Ocupados.CSV']


## 3. Catálogos de apoyo

- **Ciudades:** códigos de `AREA` que tienen territorio propio en la base.
- **Ramas de actividad:** las 13 ramas se arman agrupando las divisiones de dos dígitos de `RAMA2D_R4`. Lo que no esté en la agrupación (vacío, punto o "no sabe, no informa") queda sin rama.

In [48]:
CIUDADES = [11, 5, 76, 8, 68, 17, 66, 54, 52, 73, 41, 13, 50]

GRUPOS_RAMA = {
    1:  [1, 2, 3],                                             # Agricultura, ganadería, caza, silvicultura y pesca
    2:  [5, 6, 7, 8, 9, 35, 36, 37, 38, 39],                   # Electricidad, gas, agua y gestión de desechos
    3:  list(range(10, 34)),                                   # Industrias manufactureras
    4:  [41, 42, 43],                                          # Construcción
    5:  [45, 46, 47],                                          # Comercio y reparación de vehículos
    6:  [55, 56],                                              # Alojamiento y servicios de comida
    7:  [49, 50, 51, 52, 53],                                  # Transporte y almacenamiento
    8:  [58, 59, 60, 61, 62, 63],                              # Información y comunicaciones
    9:  [64, 65, 66],                                          # Actividades financieras y de seguros
    10: [68],                                                  # Actividades inmobiliarias
    11: [69, 70, 71, 72, 73, 74, 75, 77, 78, 79, 80, 81, 82],  # Profesionales, científicas, técnicas y administrativas
    12: [84, 85, 86, 87, 88],                                  # Administración pública, educación y salud
    13: [90, 91, 92, 93, 94, 95, 96, 97, 98],                  # Artísticas, entretenimiento y otros servicios
}
# Diccionario división -> rama
RAMA = {division: rama for rama, divisiones in GRUPOS_RAMA.items() for division in divisiones}

## 4. Funciones de apoyo

In [ ]:
def leer(modulo):
    """Lee el archivo de un módulo del mes."""
    ruta = os.path.join(CARPETA_ENTRADA, ARCHIVOS[modulo])
    try:
        t = pd.read_csv(ruta, sep=';', decimal=',', encoding='utf-8', dtype=str)
    except UnicodeDecodeError:
        t = pd.read_csv(ruta, sep=';', decimal=',', encoding='latin-1', dtype=str)
    t.columns = [c.strip().upper() for c in t.columns]
    print('Leído', ARCHIVOS[modulo], '-', len(t), 'registros')
    return t


def a_numero(columna):
    """Convierte texto a número. Lo que no se pueda convertir (vacío, '.') queda como dato faltante."""
    return pd.to_numeric(columna.str.replace(',', '.'), errors='coerce')


def armar_id(tabla):
    """id_persona = directorio (7 dígitos) + secuencia_p (2) + orden (2), con ceros a la izquierda."""
    directorio = a_numero(tabla['DIRECTORIO']).astype('Int64').astype(str).str.zfill(7)
    secuencia = a_numero(tabla['SECUENCIA_P']).astype('Int64').astype(str).str.zfill(2)
    orden = a_numero(tabla['ORDEN']).astype('Int64').astype(str).str.zfill(2)
    return directorio + secuencia + orden


reporte = []
rechazados = []


def quitar(tabla, nombre, malos, motivo):
    """Saca de la tabla los registros que cumplen 'malos', los guarda como rechazados y los cuenta en el reporte."""
    reporte.append('%-11s %-55s %5d' % (nombre, motivo, malos.sum()))
    rechazo = tabla[malos].copy()
    rechazo['tabla'] = nombre
    rechazo['motivo'] = motivo
    rechazados.append(rechazo)
    return tabla[~malos]

## 5. Lectura de los archivos del DANE

In [ ]:
personas = leer('personas')
ocupados = leer('ocupados')
no_ocupados = leer('no_ocupados')

Leído Características generales, seguridad social en salud y educación.CSV - 66909 registros
Leído Ocupados.CSV - 29691 registros
Leído No ocupados.CSV - 24114 registros


## 6. Construcción de las tres tablas

- **persona:** todas las personas de Características generales. El área se convierte en el territorio del catálogo (3 resto, el código de la ciudad si es una de las 13, 2 urbano en los demás casos).
- **ocupado:** personas del módulo Ocupados, con su rama agrupada. Los pensionados (P6920 = 3) no se cuentan como informales.
- **desocupado:** del módulo No ocupados se conservan solo los que tienen `DSI = 1` y únicamente su identificador.

In [ ]:
persona = pd.DataFrame({
    'id_persona': armar_id(personas),
    'id_periodo': id_periodo,
    'id_sexo': a_numero(personas['P3271']),
    'edad': a_numero(personas['P6040']),
    'clase': a_numero(personas['CLASE']),
    'area': a_numero(personas['AREA']),
    'factor_expansion': a_numero(personas['FEX_C18']),
})

ocupado = pd.DataFrame({
    'id_persona': armar_id(ocupados),
    'id_rama': a_numero(ocupados['RAMA2D_R4']).map(RAMA),
    'horas_trabajo': a_numero(ocupados['P6800']),
    'ingreso': a_numero(ocupados['INGLABO']),
    'cotiza_pension': a_numero(ocupados['P6920']).isin([1, 3]).astype(int),
})

desocupado = pd.DataFrame({'id_persona': armar_id(no_ocupados[a_numero(no_ocupados['DSI']) == 1])})

## 7. Verificaciones

Los registros que no pasan una verificación no se cargan: se separan en el archivo de rechazados.

In [ ]:
reporte.append('Periodo %d' % id_periodo)
reporte.append('%-11s %-55s %5s' % ('Tabla', 'Verificación (registros rechazados)', 'Casos'))
total = {'persona': len(persona), 'ocupado': len(ocupado), 'desocupado': len(desocupado)}

persona = quitar(persona, 'persona', persona['factor_expansion'].isna() | persona['id_persona'].str.contains('NA'), 'identificador o factor de expansión vacío')
persona = quitar(persona, 'persona', persona.duplicated('id_persona'), 'identificador repetido')
persona = quitar(persona, 'persona', ~persona['id_sexo'].isin([1, 2]), 'sexo distinto de 1 o 2')
persona = quitar(persona, 'persona', ~persona['edad'].between(0, 110), 'edad fuera de 0 a 110')
persona = quitar(persona, 'persona', ~persona['clase'].isin([1, 2]), 'clase distinta de 1 o 2')
persona = quitar(persona, 'persona', persona['factor_expansion'] <= 0, 'factor de expansión no positivo')

mayores_15 = persona.loc[persona['edad'] >= 15, 'id_persona']      # personas en edad de trabajar
ocupado = quitar(ocupado, 'ocupado', ocupado.duplicated('id_persona'), 'identificador repetido')
ocupado = quitar(ocupado, 'ocupado', ~ocupado['id_persona'].isin(mayores_15), 'no está en persona o es menor de 15 años')
ocupado = quitar(ocupado, 'ocupado', ocupado['horas_trabajo'] > 168, 'más de 168 horas semanales')
ocupado = quitar(ocupado, 'ocupado', ocupado['ingreso'] < 0, 'ingreso negativo')

desocupado = quitar(desocupado, 'desocupado', desocupado.duplicated('id_persona'), 'identificador repetido')
desocupado = quitar(desocupado, 'desocupado', ~desocupado['id_persona'].isin(mayores_15), 'no está en persona o es menor de 15 años')
desocupado = quitar(desocupado, 'desocupado', desocupado['id_persona'].isin(ocupado['id_persona']), 'también figura como ocupado')

reporte.append('')
reporte.append('Ocupados sin rama (no sabe, no informa o vacía): %d' % ocupado['id_rama'].isna().sum())
for nombre, tabla in [('persona', persona), ('ocupado', ocupado), ('desocupado', desocupado)]:
    reporte.append('%-11s quedan %d de %d registros' % (nombre, len(tabla), total[nombre]))
reporte.append('Población expandida del mes: %.0f' % persona['factor_expansion'].sum())

**Identificadores repetidos entre meses.** El identificador es la llave primaria de la base y no puede repetirse. Esta celda lo busca en los meses que ya están en `2. Datos para BD`.

In [ ]:
repetidos = 0
for carpeta, _, archivos in os.walk(os.path.join(RAIZ, '2. Datos para BD')):
    for archivo in archivos:
        if archivo.startswith('persona_') and archivo != 'persona_%d.csv' % id_periodo:
            anteriores = pd.read_csv(os.path.join(carpeta, archivo), usecols=['id_persona'], dtype=str)
            repetidos += persona['id_persona'].isin(anteriores['id_persona']).sum()
reporte.append('Identificadores que ya estaban en meses anteriores: %d' % repetidos)
print('\n'.join(reporte))

Periodo 202603
Tabla       Verificación (registros rechazados)                     Casos
persona     identificador o factor de expansión vacío                   0
persona     identificador repetido                                      0
persona     sexo distinto de 1 o 2                                      0
persona     edad fuera de 0 a 110                                       0
persona     clase distinta de 1 o 2                                     0
persona     factor de expansión no positivo                             0
ocupado     identificador repetido                                      0
ocupado     no está en persona o es menor de 15 años                    0
ocupado     más de 168 horas semanales                                  0
ocupado     ingreso negativo                                            0
desocupado  identificador repetido                                      0
desocupado  no está en persona o es menor de 15 años                    0
desocupado  también fig

## 8. Archivos de salida

Los datos faltantes se escriben como `\N` para que MySQL los lea como NULL.

In [ ]:
for c in ['id_periodo', 'id_sexo', 'edad', 'clase', 'area']:
    persona[c] = persona[c].astype('Int64')
ocupado['id_rama'] = ocupado['id_rama'].astype('Int64')

for nombre, tabla in [('persona', persona), ('ocupado', ocupado), ('desocupado', desocupado)]:
    ruta = os.path.join(CARPETA_SALIDA, '%s_%d.csv' % (nombre, id_periodo))
    tabla.to_csv(ruta, index=False, na_rep='\\N', lineterminator='\n')

todos_rechazados = pd.concat(rechazados)
if len(todos_rechazados) > 0:
    # Solo se crean el archivo de rechazados y el reporte cuando hubo registros rechazados
    todos_rechazados.to_csv(os.path.join(CARPETA_SALIDA, 'rechazados_%d.csv' % id_periodo), index=False)
    with open(os.path.join(CARPETA_SALIDA, 'reporte_%d.txt' % id_periodo), 'w', encoding='utf-8') as f:
        f.write('\n'.join(reporte))
    print('Hubo registros rechazados: revisar rechazados y reporte')
else:
    print('No hubo registros rechazados')

print('Archivos listos en', CARPETA_SALIDA)
print(os.listdir(CARPETA_SALIDA))

Archivos listos en C:\Users\jessi\OneDrive\Documents\GitHub\Proyecto-ingeniera-de-datos-\Entregable 2\Datos para BD\2026\Marzo
['desocupado_202603.csv', 'ocupado_202603.csv', 'persona_202603.csv', 'rechazados_202603.csv', 'reporte_202603.txt']
